####Binance trades

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

raw_root = "s3://l1l2bucket/data/raw/binance"

catalog = "crypto_lakehouse"
bronze_schema = "bronze"

spark.sql(f"create schema if not exists {catalog}.{bronze_schema}")

print(f"Raw data: {raw_root}")
print(f"Bronze schema: {catalog}.{bronze_schema}")

display(dbutils.fs.ls(raw_root))

In [ ]:
spot_trade_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("id", LongType(), True),
    StructField("orderId", LongType(), True),
    StructField("orderListId", LongType(), True),
    StructField("price", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("quoteQty", StringType(), True),
    StructField("commission", StringType(), True),
    StructField("commissionAsset", StringType(), True),
    StructField("time", LongType(), True),
    StructField("isBuyer", BooleanType(), True),
    StructField("isMaker", BooleanType(), True),
    StructField("isBestMatch", BooleanType(), True)
])

print(spot_trade_schema.simpleString())

In [ ]:
spot_trades_raw_df = (
    spark.read
    .schema(spot_trade_schema)
    .option("multiline",True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/trades/")
)

spot_trades_df = (
    spot_trades_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("exchange",lit("Binance"))
    .withColumn("source_type",lit("Spot_trade"))
    .withColumn("ingested_at", current_timestamp())
)

print("Spot trade rows:", spot_trades_df.count())

display(
    spot_trades_df.orderBy(
        col("time").desc()
    )
)

In [ ]:
spot_rows_before = spot_trades_df.count()


spot_trades_bronze_df = (
    spot_trades_df
    .dropDuplicates([
        "symbol",
        "id"
    ])
)


spot_rows_after = spot_trades_bronze_df.count()
spot_duplicates_removed = spot_rows_before - spot_rows_after


print("Rows before deduplication:", spot_rows_before)
print("Rows after deduplication:", spot_rows_after)
print("Duplicate rows removed:", spot_duplicates_removed)


display(
    spot_trades_bronze_df
    .groupBy("symbol")
    .count()
    .orderBy("symbol")
)


In [ ]:
(
    spot_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema",True)
    .saveAsTable(f"{catalog}.{bronze_schema}.binance_spot_trades")
)

####Binance Convert trade

In [ ]:
convert_trade_schema = StructType([
    StructField("quoteId", StringType(), True),
    StructField("orderId", LongType(), True),
    StructField("orderStatus", StringType(), True),
    StructField("fromAsset", StringType(), True),
    StructField("fromAmount", StringType(), True),
    StructField("toAsset", StringType(), True),
    StructField("toAmount", StringType(), True),
    StructField("ratio", StringType(), True),
    StructField("inverseRatio", StringType(), True),
    StructField("createTime", LongType(), True),
    StructField("orderType", StringType(), True),
    StructField("side", StringType(), True),
    StructField("walletType", StringType(), True)
])

print(convert_trade_schema.simpleString())

In [ ]:
convert_trades_raw_df = (
    spark.read
    .schema(convert_trade_schema)
    .option("multiline",True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/convert_trades/")
)

convert_trades_df = (
    convert_trades_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("exchange",lit("Binance"))
    .withColumn("source_type",lit("Convert_trade"))
    .withColumn("ingested_at",current_timestamp())
)

print("convert trade rows:", convert_trades_df.count())

display(
    convert_trades_df.orderBy(
        col("createTime").desc()
    )
)



In [ ]:
convert_rows_before = convert_trades_df.count()

convert_trades_bronze_df = (
    convert_trades_df
    .dropDuplicates([
        "orderId"
    ])
)

convert_rows_after = convert_trades_bronze_df.count()
duplicates_removed = convert_rows_before - convert_rows_after

print("Rows before deduplication:", convert_rows_before)
print("Rows after deduplication:", convert_rows_after)
print("Duplicate rows removed:", duplicates_removed)

display(
    convert_trades_bronze_df)

In [ ]:
(
    convert_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_convert_trades"
    )
)

In [ ]:
print(
    "Saved rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_convert_trades"
    ).count()
)
display(
    spark.table(
        f"{catalog}.{bronze_schema}.binance_convert_trades"
    )
)

####Binance p2p

In [ ]:
p2p_schema = StructType([
    StructField("orderNumber", StringType(), True),
    StructField("advNo", StringType(), True),
    StructField("tradeType", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("fiat", StringType(), True),
    StructField("fiatSymbol", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("totalPrice", StringType(), True),
    StructField("unitPrice", StringType(), True),
    StructField("orderStatus", StringType(), True),
    StructField("createTime", LongType(), True),
    StructField("takerCommissionRate", StringType(), True),
    StructField("takerCommission", StringType(), True),
    StructField("takerAmount", StringType(), True),
    StructField("counterPartNickName", StringType(), True),
    StructField("payMethodName", StringType(), True),
    StructField("additionalKycVerify", LongType(), True)
])

print(p2p_schema.simpleString())

In [ ]:
p2p_buy_df = (
    spark.read
    .schema(p2p_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup",True)
    .json(f"{raw_root}/p2p_buy/")
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
)

p2p_sell_df = (
    spark.read
    .schema(p2p_schema)
    .option("multiLine",True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/p2p_sell/")
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file"),
    )
)

p2p_trades_df = (
    p2p_buy_df
    .unionByName(p2p_sell_df)
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("P2P_trade"))
    .withColumn("ingested_at", current_timestamp())
)

print("P2P BUY rows:", p2p_buy_df.count())
print("P2P SELL rows:", p2p_sell_df.count())
print("Total P2P rows:", p2p_trades_df.count())

display(
    p2p_trades_df.orderBy(
        col("createTime").desc()
    )
)

In [ ]:
p2p_rows_before = p2p_trades_df.count()

p2p_trades_bronze_df = (
    p2p_trades_df
    .dropDuplicates([
        "orderNumber"
    ])
)

p2p_rows_after = p2p_trades_bronze_df.count()

total_duplicates_removed = p2p_rows_before - p2p_rows_after

print("P2P rows before deduplication:", p2p_rows_before)
print("P2P rows after deduplication:", p2p_rows_after)
print("Total duplicate P2P rows removed:", total_duplicates_removed)

display(
    p2p_trades_bronze_df
    .groupBy("tradeType")
    .count()
)

In [ ]:
(
    p2p_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_p2p_trades")
)

print(
    "Saved P2P rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_p2p_trades"
    ).count()
)

####Binance account

In [ ]:
commission_rates_schema = StructType([
    StructField("maker", StringType(), True),
    StructField("taker", StringType(), True),
    StructField("buyer", StringType(), True),
    StructField("seller", StringType(), True)
])


balance_schema = StructType([
    StructField("asset", StringType(), True),
    StructField("free", StringType(), True),
    StructField("locked", StringType(), True)
])


account_schema = StructType([
    StructField("makerCommission", LongType(), True),
    StructField("takerCommission", LongType(), True),
    StructField("buyerCommission", LongType(), True),
    StructField("sellerCommission", LongType(), True),
    StructField("commissionRates", commission_rates_schema, True),
    StructField("canTrade", BooleanType(), True),
    StructField("canWithdraw", BooleanType(), True),
    StructField("canDeposit", BooleanType(), True),
    StructField("brokered", BooleanType(), True),
    StructField("requireSelfTradePrevention", BooleanType(), True),
    StructField("preventSor", BooleanType(), True),
    StructField("updateTime", LongType(), True),
    StructField("accountType", StringType(), True),
    StructField(
        "balances",
        ArrayType(balance_schema),
        True
    ),
    StructField(
        "permissions",
        ArrayType(StringType()),
        True
    ),
    StructField("uid", LongType(), True)
])


print(account_schema.simpleString())

In [ ]:
account_raw_df = (
    spark.read
    .schema(account_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/account/")
)


account_df = (
    account_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Account_snapshot"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Account snapshot rows:",
    account_df.count()
)


display(
    account_df.select(
        "uid",
        "accountType",
        "canTrade",
        "canDeposit",
        "canWithdraw",
        "updateTime",
        size("balances").alias("number_of_assets"),
        "source_file_name",
        "ingested_at"
    )
)

In [ ]:
account_rows_before = account_df.count()


account_bronze_df = (
    account_df
    .dropDuplicates([
        "_source_file"
    ])
)

account_rows_after = account_bronze_df.count()


print(
    "Account rows before deduplication:",
    account_rows_before
)

print(
    "Account rows after deduplication:",
    account_rows_after
)

print(
    "Duplicate account rows removed:",
    account_rows_before - account_rows_after
)

In [ ]:
(
    account_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_account_snapshots"
    )
)

print(
    "Saved account snapshots:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_account_snapshots"
    ).count()
)

####Binance Deposits

In [ ]:
deposit_schema = StructType([
    StructField("id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("coin", StringType(), True),
    StructField("network", StringType(), True),
    StructField("status", LongType(), True),
    StructField("address", StringType(), True),
    StructField("addressTag", StringType(), True),
    StructField("txId", StringType(), True),
    StructField("sourceAddress", StringType(), True),
    StructField("insertTime", LongType(), True),
    StructField("completeTime", LongType(), True),
    StructField("transferType", LongType(), True),
    StructField("confirmTimes", StringType(), True),
    StructField("unlockConfirm", LongType(), True),
    StructField("walletType", LongType(), True),
    StructField("travelRuleStatus", LongType(), True)
])


print(deposit_schema.simpleString())

In [ ]:
deposits_raw_df = (
    spark.read
    .schema(deposit_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/deposits/")
)


deposits_df = (
    deposits_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Deposit"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Deposit rows:",
    deposits_df.count()
)


display(
    deposits_df.orderBy(
        col("insertTime").desc()
    )
)

In [ ]:
deposits_bronze_df = (
    deposits_df
    .dropDuplicates([
        "id"
    ])
)


(
    deposits_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_deposits"
    )
)


print(
    "Saved deposit rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_deposits"
    ).count()
)

####Binance withdrawals

In [ ]:
withdrawal_schema = StructType([
    StructField("id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("transactionFee", StringType(), True),
    StructField("coin", StringType(), True),
    StructField("status", LongType(), True),
    StructField("address", StringType(), True),
    StructField("txId", StringType(), True),
    StructField("applyTime", StringType(), True),
    StructField("network", StringType(), True),
    StructField("transferType", LongType(), True),
    StructField("withdrawOrderId", StringType(), True),
    StructField("info", StringType(), True),
    StructField("confirmNo", LongType(), True),
    StructField("walletType", LongType(), True),
    StructField("txKey", StringType(), True),
    StructField("completeTime", StringType(), True)
])


print(withdrawal_schema.simpleString())

In [ ]:
withdrawals_raw_df = (
    spark.read
    .schema(withdrawal_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/withdrawals/")
)


withdrawals_df = (
    withdrawals_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Withdrawal"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Withdrawal rows:",
    withdrawals_df.count()
)


display(
    withdrawals_df.orderBy(
        col("applyTime").desc()
    )
)

In [ ]:
withdrawals_bronze_df = (
    withdrawals_df
    .dropDuplicates([
        "id"
    ])
)


print(
    "Duplicate withdrawal rows removed:",
    withdrawals_df.count() - withdrawals_bronze_df.count()
)


(
    withdrawals_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_withdrawals"
    )
)


print(
    "Saved withdrawal rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_withdrawals"
    ).count()
)

####Binance transfers

In [ ]:
transfer_schema = StructType([
    StructField("tranId", LongType(), True),
    StructField("asset", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("transferType", StringType(), True)
])


print(transfer_schema.simpleString())

In [ ]:
transfers_raw_df = (
    spark.read
    .schema(transfer_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/transfers/")
)


transfers_df = (
    transfers_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Binance"))
    .withColumn("source_type", lit("Transfer"))
    .withColumn("ingested_at", current_timestamp())
)


print(
    "Transfer rows:",
    transfers_df.count()
)


display(
    transfers_df.orderBy(
        col("timestamp").desc()
    )
)

In [ ]:
transfers_bronze_df = (
    transfers_df
    .dropDuplicates([
        "tranId"
    ])
)


print(
    "Duplicate transfer rows removed:",
    transfers_df.count() - transfers_bronze_df.count()
)


(
    transfers_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_transfers"
    )
)


print(
    "Saved transfer rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_transfers"
    ).count()
)

####Binance price history

In [ ]:
kline_file_schema = ArrayType(
    ArrayType(
        StringType()
    )
)


kline_field_names = [
    "open_time",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "unused_field"
]


print(kline_file_schema.simpleString())
print(kline_field_names)

In [ ]:
price_files_df = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", True)
    .load(f"{raw_root}/price_history/")
)


price_klines_raw_df = (
    price_files_df
    .withColumn(
        "symbol",
        regexp_extract(
            col("path"),
            r"/price_history/([^/]+)/",
            1
        )
    )
    .withColumn(
        "klines",
        from_json(
            decode(col("content"), "UTF-8"),
            kline_file_schema
        )
    )
    .select(
        "symbol",

        col("path").alias("_source_file"),

        to_timestamp(
            regexp_extract(
                col("path"),
                r"price_history_(\d{8}T\d{6}_\d{6}Z)_",
                1
            ),
            "yyyyMMdd'T'HHmmss_SSSSSS'Z'"
        ).alias("source_file_saved_at"),

        col("modificationTime").alias(
            "source_file_modified_at"
        ),

        explode("klines").alias("kline")
    )
)


print(
    "Candlestick rows:",
    price_klines_raw_df.count()
)


display(
    price_klines_raw_df.select(
        "symbol",
        "_source_file",
        "source_file_saved_at",
        "source_file_modified_at"
    ).limit(10)
)

In [ ]:
price_files_df.printSchema()

In [ ]:
kline_columns = [
    col("kline")[index].alias(name)
    for index, name in enumerate(kline_field_names)
]


price_history_df = (
    price_klines_raw_df
    .select(
        "symbol",
        *kline_columns,
        "_source_file",
        "source_file_saved_at",
        "source_file_modified_at"
    )
    .withColumn(
        "exchange",
        lit("Binance")
    )
    .withColumn(
        "source_type",
        lit("Daily_price")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
)


print(
    "Price-history rows:",
    price_history_df.count()
)

print(
    "Rows missing file save time:",
    price_history_df
    .filter(col("source_file_saved_at").isNull())
    .count()
)

In [ ]:
from pyspark.sql.window import Window


latest_price_file_window = (
    Window
    .partitionBy(
        "symbol",
        "open_time"
    )
    .orderBy(
        col("source_file_saved_at").desc(),
        col("_source_file").desc()
    )
)


price_history_bronze_df = (
    price_history_df
    .withColumn(
        "row_number",
        row_number().over(latest_price_file_window)
    )
    .filter(col("row_number") == 1)
    .drop("row_number")
)


print(
    "Duplicate candles removed:",
    price_history_df.count()
    - price_history_bronze_df.count()
)


(
    price_history_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_price_history"
    )
)


print(
    "Saved candlestick rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.binance_price_history"
    ).count()
)

####Binance funding wallet

In [ ]:
funding_balance_schema = StructType([
    StructField("asset", StringType(), True),
    StructField("free", StringType(), True),
    StructField("locked", StringType(), True),
    StructField("freeze", StringType(), True),
    StructField("withdrawing", StringType(), True),
    StructField("btcValuation", StringType(), True)
])

funding_account_schema = StructType([
    StructField("walletType", StringType(), True),
    StructField("capturedAt", StringType(), True),

    StructField(
        "balances",
        ArrayType(
            funding_balance_schema,
            True
        ),
        True
    )
])

In [ ]:
funding_account_raw_df = (
    spark.read
    .schema(funding_account_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(
        f"{raw_root}/funding_account/"
    )
)

funding_account_bronze_df = (
    funding_account_raw_df
    .select(
        "*",

        col("_metadata.file_name").alias(
            "source_file_name"
        ),

        col("_metadata.file_path").alias(
            "_source_file"
        )
    )
    .withColumn(
        "exchange",
        lit("Binance")
    )
    .withColumn(
        "source_type",
        lit("Funding_account")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
)

print(
    "Funding account snapshot rows:",
    funding_account_bronze_df.count()
)

funding_account_bronze_df.printSchema()

display(funding_account_bronze_df)

In [ ]:
(
    funding_account_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{bronze_schema}.binance_funding_account_snapshots"
    )
)

saved_funding_rows = spark.table(
    f"{catalog}.{bronze_schema}.binance_funding_account_snapshots"
).count()

print(
    "Saved Funding account snapshots:",
    saved_funding_rows
)

In [ ]:
bronze_tables = [
    "binance_spot_trades",
    "binance_convert_trades",
    "binance_p2p_trades",
    "binance_account_snapshots",
    "binance_deposits",
    "binance_withdrawals",
    "binance_transfers",
    "binance_price_history",
    "binance_funding_account_snapshots"
]


for table_name in bronze_tables:

    full_table_name = (
        f"{catalog}.{bronze_schema}.{table_name}"
    )

    row_count = (
        spark.table(full_table_name)
        .count()
    )

    print(
        f"{table_name}: {row_count} rows"
    )